# 02 — Pre-game Features & Baselines

Goal: predict `watchability_score` (from notebook 01) using **only what's known before kickoff**.

- **Features** (`src/features.py`): Vegas spread and over/under, playoff stakes, division game, primetime, rest, weather, and each team's form over its **previous** 8 games (`shift(1)` so a game never sees its own result — that would leak the label).
- **Validation:** walk-forward (Phase 2 — Time Series). For each test season S in 2016–2025: train on seasons < S, test on S.
- **Models, in order:** predict-the-mean → Ridge → LightGBM. Without the floor you can't tell whether the boosted model is actually good.

In [1]:
import sys; sys.path.insert(0, '../src')
import numpy as np, pandas as pd, lightgbm as lgb
from scipy.stats import spearmanr
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from features import build_features

master = pd.read_csv('../data/master_games.csv', low_memory=False)
labels = pd.read_csv('../data/labeled_games.csv', usecols=['game_id', 'watchability_score'])
games = master.merge(labels, on='game_id', how='left')

X_all = build_features(games).merge(labels, on='game_id', how='left')
data = X_all[X_all['watchability_score'].notna()].reset_index(drop=True)
NUM = [c for c in data.columns if c not in ('game_id', 'season', 'roof', 'watchability_score')]
CAT = ['roof']
print('Labeled rows:', len(data), '| features:', len(NUM) + len(CAT))
print('Most-missing features:'); print(data[NUM].isna().mean().sort_values(ascending=False).head(4).round(3).to_string())

Labeled rows: 7047 | features: 21
Most-missing features:
temp                  0.287
wind                  0.287
gap_season_win_pct    0.061
sum_season_win_pct    0.061


**Missing values:** `temp`/`wind` are blank for domes, and form features are blank for a team's first game in the data. The Ridge pipeline uses median imputation **plus indicator columns** (Phase 2 — Feature Engineering) so "missing" can itself carry signal. LightGBM handles NaN natively by learning which branch missing values should go down.

Everything sits inside a `Pipeline`/`ColumnTransformer`, so imputation medians and scaler means are fit on the training seasons only (Phase 2 — ML Pipelines). `handle_unknown='ignore'` covers a new `roof` value showing up in a future season.

In [2]:
def ridge_model():
    pre = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median', add_indicator=True)),
                          ('sc', StandardScaler())]), NUM),
        ('cat', OneHotEncoder(handle_unknown='ignore'), CAT)])
    return Pipeline([('pre', pre), ('ridge', Ridge(alpha=10.0))])

def lgb_fit_predict(train, test):
    # Early stopping needs data the model doesn't train on: hold out the last training season.
    # (Phase 2 — Hyperparameter Tuning: stop on validation loss instead of tuning n_estimators.)
    val_season = train['season'].max()
    tr, va = train[train.season < val_season], train[train.season == val_season]
    prep = lambda d: d[NUM + CAT].assign(roof=d['roof'].astype(pd.CategoricalDtype(sorted(data.roof.dropna().unique()))))
    m = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.03, num_leaves=15, min_child_samples=40,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)
    m.fit(prep(tr), tr['watchability_score'], eval_X=(prep(va),), eval_y=(va['watchability_score'],),
          callbacks=[lgb.early_stopping(100, verbose=False)])
    return m.predict(prep(test)), m

def week_spearman(df, pred):
    # The real question is "which game THIS week?" -> rank agreement within each week.
    rs = [spearmanr(g['watchability_score'], pred[g.index]).statistic
          for _, g in df.groupby('week') if len(g) >= 3]
    return np.nanmean(rs)

## Walk-forward evaluation

In [3]:
rows, preds = [], {}
for S in range(2016, 2026):
    train, test = data[data.season < S], data[data.season == S]
    y_tr, y_te = train['watchability_score'], test['watchability_score']
    p_mean = pd.Series(y_tr.mean(), index=test.index)
    p_ridge = pd.Series(ridge_model().fit(train, y_tr).predict(test), index=test.index)
    p_lgb, lgb_m = lgb_fit_predict(train, test); p_lgb = pd.Series(p_lgb, index=test.index)
    for name, p in [('mean', p_mean), ('ridge', p_ridge), ('lgbm', p_lgb)]:
        rows.append({'season': S, 'model': name,
                     'rmse': mean_squared_error(y_te, p) ** 0.5, 'r2': r2_score(y_te, p),
                     'week_spearman': week_spearman(test, p) if name != 'mean' else 0.0})
res = pd.DataFrame(rows)
res.groupby('model')[['rmse', 'r2', 'week_spearman']].mean().round(3)

,rmse,r2,week_spearman
model,,,
lgbm,14.349,0.075,0.169
mean,14.945,-0.003,0.000
ridge,14.333,0.077,0.189


In [4]:
res.pivot(index='season', columns='model', values='r2').round(3)

model,lgbm,mean,ridge
season,,,
2016,0.090,-0.000,0.067
2017,0.056,-0.010,0.048
2018,0.053,-0.001,0.060
2019,0.049,-0.001,0.067
2020,0.080,-0.003,0.082
2021,0.113,-0.004,0.123
2022,0.043,-0.005,0.040
2023,0.062,-0.007,0.062
2024,0.115,-0.000,0.123


## What is the model leaning on?

LightGBM gain importance from the final fold (trained through 2024). Gain is MDI-style (Phase 2 — Random Forests), so it's biased toward continuous features and doesn't correct for correlated ones. Treat it as a first look; permutation importance is the check to run later.

In [5]:
pd.Series(lgb_m.booster_.feature_importance('gain'), index=lgb_m.booster_.feature_name()) \
  .sort_values(ascending=False).head(8).round(0)

playoff_stakes    603657.0
abs_spread        497951.0
div_game          403578.0
sum_form_y        353484.0
total_line        323589.0
gap_form_y        323015.0
gap_form_pf       319857.0
week              299053.0
dtype: float64

## Takeaways (baseline run, 2016–2025 walk-forward)

| model | RMSE | R² | within-week Spearman |
|---|---|---|---|
| predict mean | 14.95 | 0.00 | 0.00 |
| Ridge | 14.33 | 0.077 | 0.19 |
| LightGBM | 14.35 | 0.075 | 0.17 |

- **Both models beat the floor, but only slightly.** Ridge ≈ LightGBM, so there's no non-linear structure for boosting to exploit.
- **Why R² fell from 0.25 to 0.08 when rivalry went 0.15 → 0.05:** `div_game` is a pre-game feature that was copied straight into the label at 15%. The model "predicted" that slice perfectly, which inflated R². With rivalry at 5%, the label is now mostly in-game drama, which is barely predictable (see below). **The model didn't get worse; the target got more honest.** The lesson is that R² is only meaningful relative to what's in `y`.
- The remaining signal comes from playoff stakes (known in advance), Vegas closeness (`abs_spread`), and expected scoring (`total_line`).

## How predictable is each drama ingredient?

The label is `0.30·excitement + 0.15·comeback + 0.15·fireworks + 0.25·stakes + 0.15·rivalry`. Stakes and rivalry are known before kickoff, so the open question is the other three. Here each one (percentile-scaled, exactly as in the label) becomes its own target, using the same features and walk-forward loop.

This is the **ceiling check**: if pre-game features can't predict an ingredient, no amount of tuning on the combined score will fix that part.

In [6]:
lab = pd.read_csv('../data/labeled_games.csv', usecols=['game_id', 'excitement_weighted', 'comeback_factor', 'fireworks'])
for c in ['excitement_weighted', 'comeback_factor', 'fireworks']:
    lab[c] = lab[c].rank(pct=True)
ing = data.merge(lab, on='game_id', how='left')

rows = []
for target in ['excitement_weighted', 'comeback_factor', 'fireworks']:
    d = ing.assign(watchability_score=ing[target])   # reuse helpers: they read 'watchability_score'
    for S in range(2016, 2026):
        tr, te = d[d.season < S], d[d.season == S]
        for name, p in [('ridge', ridge_model().fit(tr, tr.watchability_score).predict(te)),
                        ('lgbm', lgb_fit_predict(tr, te)[0])]:
            rows.append({'target': target, 'model': name, 'r2': r2_score(te.watchability_score, p)})
pd.DataFrame(rows).pivot_table(index='target', columns='model', values='r2').round(3)

model,lgbm,ridge
target,,
comeback_factor,0.003,-0.003
excitement_weighted,0.017,0.015
fireworks,0.073,0.077


In [7]:
# Which single pre-game feature tracks each ingredient? (Spearman, all labeled games)
feats = ['total_line', 'abs_spread', 'sum_form_pf', 'sum_form_y', 'min_win_pct']
ing[['excitement_weighted', 'comeback_factor', 'fireworks'] + feats].corr(method='spearman') \
   .loc[feats, ['excitement_weighted', 'comeback_factor', 'fireworks']].round(3)

,excitement_weighted,comeback_factor,fireworks
total_line,-0.033,-0.007,0.314
abs_spread,-0.121,-0.117,0.028
sum_form_pf,-0.038,-0.028,0.202
sum_form_y,-0.009,0.026,0.069
min_win_pct,0.020,-0.002,0.066


### Takeaways: the drama is (almost) unpredictable before kickoff

| ingredient | label weight | best walk-forward R² |
|---|---|---|
| excitement_weighted | 0.335 | 0.016 |
| comeback_factor | 0.17 | 0.003 |
| fireworks | 0.17 | 0.077 |

- **~50% of the label (excitement + comeback) is essentially noise from a pre-game point of view.** The best single signal is `abs_spread` (Spearman −0.12): games Vegas expects to be close are slightly more dramatic. That's all there is.
- **`fireworks` is somewhat predictable.** `total_line` has Spearman 0.31 with it, and recent scoring form 0.20. That fits: the over/under *is* the market's scoring forecast.
- **So the combined model's R² comes almost entirely from the ingredients known before kickoff** (stakes, rivalry) plus a little from fireworks. That's the ceiling this feature set can reach, not a bug in the model.
- **This doesn't make the drama part of the label wrong.** The model predicts the *expected* watchability, E[y | pre-game info]. Randomness only lowers R²; it doesn't bias the ranking. It does mean more team-form features will have limited payoff, because they mostly target the unpredictable 45%.

## Experiment: can a targeted feature batch move the ceiling?

One timeboxed run. Three new feature groups (`extra_features` in `src/features.py`), all computed only from earlier games:
- **Team drama history:** both teams' average excitement and comeback percentiles over their last 8 games. The hypothesis: some teams (defense-light, late-game-chaos teams) are consistently dramatic.
- **Continuity:** is either team starting a different QB than last game? How many games has the least-tenured head coach coached (log-scaled)?
- **Head-to-head:** watchability score of the last meeting between these two teams.

The comparison is the same walk-forward loop, **base vs. base + extra**. Deciding rule: if within-week Spearman doesn't improve by a clear margin (say +0.02), the batch isn't worth keeping.

**Multiple-comparisons caution (Phase 1 — Statistics):** testing many feature ideas against the same test seasons turns the test set into a tuning set. That's why this is one pre-declared batch, not an open-ended search.

In [8]:
from features import extra_features
lab2 = pd.read_csv('../data/labeled_games.csv', usecols=['game_id', 'watchability_score'])
ex = extra_features(master.merge(lab2, on='game_id', how='left'))
EXTRA = [c for c in ex.columns if c != 'game_id']
data_x = data.merge(ex, on='game_id', how='left')
print(data_x[EXTRA].describe().loc[['mean', 'min', 'max']].round(2))

BASE_NUM = NUM.copy()
rows = []
for label, cols in [('base', BASE_NUM), ('base+extra', BASE_NUM + EXTRA)]:
    NUM = cols   # helpers read the global NUM
    for S in range(2016, 2026):
        tr, te = data_x[data_x.season < S], data_x[data_x.season == S]
        for name, p in [('ridge', ridge_model().fit(tr, tr.watchability_score).predict(te)),
                        ('lgbm', lgb_fit_predict(tr, te)[0])]:
            p = pd.Series(p, index=te.index)
            rows.append({'features': label, 'model': name, 'r2': r2_score(te.watchability_score, p),
                         'week_spearman': week_spearman(te, p)})
NUM = BASE_NUM
cmp_ = pd.DataFrame(rows).groupby(['model', 'features'])[['r2', 'week_spearman']].mean().round(3)
cmp_

      sum_form_excite  sum_form_comeback  any_qb_change  min_coach_games  \
mean             1.00               1.00           0.21             2.92   
min              0.30               0.30           0.00             0.00   
max              1.83               1.98           1.00             5.76   

      h2h_last_y  
mean       36.34  
min         2.57  
max        87.46  


r2  week_spearman
model features                        
lgbm  base        0.075          0.169
      base+extra  0.075          0.183
ridge base        0.077          0.189
      base+extra  0.077          0.185

In [9]:
# Per-feature signal of the new batch (Spearman with the label)
data_x[EXTRA + ['watchability_score']].corr(method='spearman')['watchability_score'].drop('watchability_score').round(3)

sum_form_excite     -0.010
sum_form_comeback    0.006
any_qb_change       -0.017
min_coach_games      0.019
h2h_last_y           0.014
Name: watchability_score, dtype: float64

### Result: rejected

| model | features | R² | within-week Spearman |
|---|---|---|---|
| Ridge | base | 0.077 | 0.189 |
| Ridge | base+extra | 0.077 | 0.185 |
| LightGBM | base | 0.075 | 0.169 |
| LightGBM | base+extra | 0.075 | 0.183 |

- Neither model clears the pre-declared +0.02 bar. Ridge, the best model, doesn't move.
- **Team drama history has ~0 correlation with the label** (−0.010, 0.006). A team's past excitement doesn't carry into its next game. That's more evidence that drama is game-level randomness, not a team trait.
- **`h2h_last_y` confirms the proxy story.** Under the old rivalry weight (0.15) it had Spearman 0.20 with the label. After cutting rivalry to 0.05, it dropped to **0.014**. Its apparent signal was almost entirely "these two teams are division rivals," which `div_game` already captures. This is the univariate-vs-multivariate credit trap from Phase 1, shown directly.

**Decision:** keep the base feature set with Ridge. `extra_features` stays in `src/features.py` as documentation of what was tried.